# Final 5-minute point model

The final point model is LightGBM, fit only on the training window. It uses the same ex-ante fields as the ridge baseline. Calendar fields are categories (`hour`, `dow`, `month`), not integer trends. `year` is omitted because the test year is outside training.

The tree is fit under MAE to the 5-minute price change, `MCP_t - MCP_{t-5min}`. The issued forecast adds that change to the lagged price, so 5-minute persistence is the zero-change forecast. MAE matches the selection metric on a price with very heavy tails.

Hyperparameters are chosen by inner chronological validation on July-September 2025, using MAE of the reconstructed price. The search does not see calibration or test. The selected parameters are then refit on all training rows. The saved model is `models/lgbm_5min_ahead.joblib`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd().resolve()
if not (_root / "scripts").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import joblib

from scripts.experiment import MODEL_PATH, load_ready
from scripts.forecast_design import TARGET, TREE_FEATURES, slice_split
from scripts.metrics import block_bootstrap_mae_diff, monthly_mae, point_scores, skill_mae
from scripts.point_models import persistence, predict_lgbm, select_lgbm

ready = load_ready()
train = slice_split(ready, "train")
test = slice_split(ready, "test")
bundle = select_lgbm(train, TREE_FEATURES, TARGET)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"bundle": bundle, "features": TREE_FEATURES}, MODEL_PATH)
print(bundle["grid"].to_string(index=False))
print("selected", bundle["params"], "inner MAE", round(bundle["inner_mae"], 4))
print("inner persistence MAE", round(bundle["persistence_inner_mae"], 4))
print("saved", MODEL_PATH)

In [ ]:
yhat = predict_lgbm(bundle, test, TREE_FEATURES)
persist = persistence(test, "mcp_lag_5min")
scores = point_scores(test[TARGET], yhat)
persist_scores = point_scores(test[TARGET], persist)
print("LightGBM test", scores)
print("persistence test", persist_scores)
print("MAE skill vs persistence", round(skill_mae(scores["mae"], persist_scores["mae"]), 4))
print(block_bootstrap_mae_diff(test[TARGET], yhat, persist))

In [ ]:
spike_cut = float(train[TARGET].quantile(0.95))
floor_cut = float(train[TARGET].quantile(0.05))
tail = (test[TARGET] >= spike_cut) | (test[TARGET] <= floor_cut)
print(f"tail cuts from training only: {floor_cut:.2f}, {spike_cut:.2f}; test tail share {tail.mean():.3%}")
regime = pd.DataFrame([
    {"regime": "body_lightgbm", **point_scores(test.loc[~tail, TARGET], yhat.loc[~tail])},
    {"regime": "body_persistence", **point_scores(test.loc[~tail, TARGET], persist.loc[~tail])},
    {"regime": "tail_lightgbm", **point_scores(test.loc[tail, TARGET], yhat.loc[tail])},
    {"regime": "tail_persistence", **point_scores(test.loc[tail, TARGET], persist.loc[tail])},
])
display(regime)
monthly = monthly_mae(test.index, test[TARGET], yhat).merge(
    monthly_mae(test.index, test[TARGET], persist).rename(columns={"mae": "persistence_mae"})[["month", "persistence_mae"]],
    on="month",
)
monthly

## Reading the point scores

Skill is `1 - MAE_model / MAE_persistence` on the same test origins. A positive skill means the model beats 5-minute persistence on MAE. The block bootstrap uses day-length blocks because the forecast errors are serially dependent; the interval is for the MAE difference, not a conformal interval.

Tail rows are defined by training quantiles of MCP, then applied to test outcomes. That stratification is evaluative. It is not used to choose the model.